# Stable Audio ControlNet - Low Resource Inference

This notebook is optimized for **4GB VRAM GPUs** like RTX 3050.

**Key optimizations:**
- Minimal model size (depth_factor=0.1)
- Fewer diffusion steps (25 instead of 100)
- Short audio clips (10 seconds)
- Memory management and cleanup

**Requirements:**
- RTX 3050 (4GB VRAM) or better
- 8GB RAM minimum
- All dependencies installed from requirements.txt


## 1. Setup and Imports


In [ ]:
import os
import gc
import sys

import torch
import torchaudio
from IPython.display import Audio, display

# Add parent directory to path
sys.path.insert(0, os.path.abspath('..'))

from stable_audio_tools.inference.generation import generate_diffusion_cond
from main.controlnet.pretrained import get_pretrained_controlnet_model

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 2. Configuration

Adjust these parameters based on your hardware:
- **depth_factor**: 0.05-0.1 for 4GB, 0.2 for 6GB+, 0.5 for 16GB+
- **steps**: 10-25 for 4GB, 50-100 for 8GB+
- **max_duration**: 5-10 seconds for 4GB, 20-30 for 8GB+


In [ ]:
# Model configuration
DEPTH_FACTOR = 0.1  # Minimal model size
CHECKPOINT_PATH = None  # Set to checkpoint path if available

# Generation parameters
SEED = 42
STEPS = 25  # Fewer steps = faster, less memory
CFG_SCALE = 5.0  # Lower than default 7.0
MAX_DURATION = 10.0  # Maximum audio duration in seconds

# Audio settings
SAMPLE_RATE = 44100

# Input/Output
INPUT_AUDIO_PATH = "../res/track_musdb/input.wav"  # Change to your audio
OUTPUT_DIR = "outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)


## 3. Memory Management Functions


In [ ]:
def clear_memory():
    """Clear GPU and system memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def print_memory_usage():
    """Print current GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"GPU Memory - Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB")
    else:
        print("CUDA not available")

clear_memory()
print_memory_usage()


## 4. Load Model

This will download Stable Audio Open weights if not already cached.


In [ ]:
print(f"Loading model with depth_factor={DEPTH_FACTOR}...")

# Load base model
model, model_config = get_pretrained_controlnet_model(
    "stabilityai/stable-audio-open-1.0",
    controlnet_types=["audio"],
    depth_factor=DEPTH_FACTOR
)

# Load checkpoint if provided
if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
    print(f"Loading checkpoint from {CHECKPOINT_PATH}...")
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
    state_dict = ckpt.get('state_dict', ckpt)
    
    # Remove 'model.' prefix if present
    new_state_dict = {}
    for k, v in state_dict.items():
        new_state_dict[k[6:] if k.startswith('model.') else k] = v
    
    model.load_state_dict(new_state_dict, strict=False)
    print("✅ Checkpoint loaded!")
else:
    print("⚠️ No checkpoint provided, using base model")

# Move to GPU and freeze unnecessary parts
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

model.model.model.requires_grad_(False)
model.conditioner.requires_grad_(False)
model.pretransform.requires_grad_(False)

clear_memory()
print("✅ Model loaded!")
print_memory_usage()


## 5. Load Input Audio


In [ ]:
print(f"Loading audio from {INPUT_AUDIO_PATH}...")

# Load audio
audio, sr = torchaudio.load(INPUT_AUDIO_PATH)

# Resample if needed
if sr != SAMPLE_RATE:
    print(f"Resampling from {sr}Hz to {SAMPLE_RATE}Hz...")
    resampler = torchaudio.transforms.Resample(sr, SAMPLE_RATE)
    audio = resampler(audio)

# Convert to mono if stereo
if audio.shape[0] > 1:
    audio = audio.mean(dim=0, keepdim=True)

# Trim to max duration
max_samples = int(MAX_DURATION * SAMPLE_RATE)
if audio.shape[1] > max_samples:
    print(f"Trimming audio to {MAX_DURATION} seconds...")
    audio = audio[:, :max_samples]

# Normalize
audio = torch.clamp(audio, -1, 1)

print(f"Audio shape: {audio.shape}")
print(f"Duration: {audio.shape[-1] / SAMPLE_RATE:.2f} seconds")

# Display input audio
display(Audio(audio.numpy(), rate=SAMPLE_RATE))


## 6. Generate Audio

This is where the magic happens! 🎵


In [ ]:
print(f"Generating audio with {STEPS} steps...")
print("This may take a few minutes on 4GB VRAM...")

# Set seed
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Prepare conditioning
conditioning_audio = audio.to(device)
prompt = ""  # Add your prompt here if desired

conditioning = [{
    "audio": conditioning_audio,
    "prompt": prompt,
    "seconds_start": 0,
    "seconds_total": conditioning_audio.shape[-1] / SAMPLE_RATE,
}]

print_memory_usage()

# Generate
try:
    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=True):
            output = generate_diffusion_cond(
                model.model,
                seed=SEED,
                batch_size=1,
                steps=STEPS,
                cfg_scale=CFG_SCALE,
                conditioning=conditioning,
                sample_size=conditioning_audio.shape[-1],
                sigma_min=0.3,
                sigma_max=500,
                sampler_type="dpmpp-3m-sde",
                device=device
            )
    
    print("✅ Generation successful!")
    if torch.cuda.is_available():
        print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    
except RuntimeError as e:
    if "out of memory" in str(e):
        print("\n❌ OUT OF MEMORY ERROR!")
        print("\nTry: Reduce DEPTH_FACTOR, STEPS, or MAX_DURATION")
        raise
    else:
        raise

clear_memory()


## 7. Listen to Results


In [ ]:
print("Input Audio:")
display(Audio(audio.cpu().numpy(), rate=SAMPLE_RATE))

print("\nGenerated Audio:")
display(Audio(output[0].cpu().numpy(), rate=SAMPLE_RATE))

print("\nMixed Audio (Input + Output):")
mixed = audio + output[0].cpu()
display(Audio(mixed.numpy(), rate=SAMPLE_RATE))


## 8. Save Results


In [ ]:
torchaudio.save(os.path.join(OUTPUT_DIR, "input.wav"), audio.cpu(), SAMPLE_RATE)
torchaudio.save(os.path.join(OUTPUT_DIR, "output.wav"), output[0].cpu(), SAMPLE_RATE)
torchaudio.save(os.path.join(OUTPUT_DIR, "mix.wav"), mixed, SAMPLE_RATE)

print(f"✅ Saved to {OUTPUT_DIR}/")
